UCS420 (Cognitive Computing) - Assignment 4

Question 1

In [2]:
import pandas as pd

roll_number = input("Enter Roll Number: ").strip()
last_2_digits = [int(d) for d in roll_number[-2:]]

categories = ["billing","account","general"]

fixed_entries = [
{"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
"keywords": "fee cost price charge", "category": "billing"},
{"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
"keywords": "password reset login", "category": "account"},
{"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
"keywords": "hours timing open time", "category": "general"},
{"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
"keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = []

question_templates = {
    "billing": [
        (
            "how can i check my fee payment status",
            "You can check your fee payment status from the billing section.",
            "fee payment status billing"
        ),
        (
            "how can i get a fee receipt",
            "You can download your fee receipt from the billing section.",
            "fee receipt billing download"
        )
    ],
    "account": [
        (
            "how do i update my registered mobile number",
            "Go to Account Settings > Personal Information to update your mobile number.",
            "mobile number update account"
        ),
        (
            "how do i change my registered email",
            "Go to Account Settings > Personal Information to change your email.",
            "email change update account"
        )
    ],
    "general": [
        (
            "where can i get general information",
            "You can find general information in the Help section.",
            "information help general"
        ),
        (
            "how can i contact customer support",
            "You can contact customer support through the Help section.",
            "support contact help"
        )
    ]
}

for digit in last_2_digits :
  category = categories[digit % 3]
  question, answer, keywords = question_templates[category][0]

  personalized_entries.append({"question": question, "answer": answer, "keywords": keywords, "category": category})

faq_entries = fixed_entries + personalized_entries

df = pd.DataFrame(faq_entries)

print("FInal 6-row Data Frame")
display(df)

Enter Roll Number: 1024170197
FInal 6-row Data Frame


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status from the...,fee payment status billing,billing
5,how do i update my registered mobile number,Go to Account Settings > Personal Information ...,mobile number update account,account


Question 2

In [5]:
def score_query (query, df) :
  query_words = set(query.lower().split())
  results = []

  for index, row in df.iterrows() :
    keywords = set(row["keywords"].lower().split())
    matches = query_words.intersection(keywords)
    score = len(matches)

    if score > 0 :
      results.append({
          "question" : row["question"],
          "answer" : row["answer"],
          "category" : row["category"],
          "matched_keywords" : ", " .join(matches),
          "confidence" : score
      })

  result_df = pd.DataFrame(results)

  if result_df.empty :
    print("No matching FAQ")
    return result_df

  result_df = result_df.sort_values(
      by = "confidence",
      ascending = False
  ).reset_index(drop = True)

  return result_df

In [6]:
query = input("Enter Question: ")
result = score_query (query, df)
display(result)

Enter Question: how can i pay the fee


,question,answer,category,matched_keywords,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,"fee, pay",2
1,what is the annual fee,The annual fee is Rs 500.,billing,fee,1
2,how can i check my fee payment status,You can check your fee payment status from the...,billing,fee,1


Question 3

In [7]:
def same_category (category_name, df) :
  return df[df["category"].str.lower() == category_name.lower()][
      ["question", "answer", "keywords", "category"]
  ]

In [8]:
personalized_category = personalized_entries[0]["category"]
print("Personalized Entry Category: ", personalized_category)
category_result = same_category(personalized_category, df)
display(category_result)

Personalized Entry Category:  billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status from the...,fee payment status billing,billing


Question 4

In [9]:
print("Available FAQ Entries: ")

for i, question in enumerate(df["question"]) :
  print(i, ":", question)

entry_number = int(input("Enter the entry number to update:"))
new_keyword = input("Enter a new keyword: ").strip().lower()

existing_keywords = df.loc[entry_number, "keywords"].split()

if new_keyword not in existing_keywords :
  existing_keywords.append(new_keyword)

df.loc[entry_number, "keywords"] = " ".join(existing_keywords)

file_name = f"1024170197_FAQ_Data.csv"

df.to_csv(file_name, index = False)

print("Updated Dataframe: ")
display(df)

print("File saved as: ", file_name)

Available FAQ Entries: 
0 : what is the annual fee
1 : how to reset password
2 : what are your working hours
3 : how can i pay the fee
4 : how can i check my fee payment status
5 : how do i update my registered mobile number
Enter the entry number to update:4
Enter a new keyword: how to check fee payment status
Updated Dataframe: 


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status from the...,fee payment status billing how to check fee pa...,billing
5,how do i update my registered mobile number,Go to Account Settings > Personal Information ...,mobile number update account,account


File saved as:  1024170197_FAQ_Data.csv


Question 5

In [10]:
category_counts = df.groupby("category").size()

print("Number of FAQ entries per category: ")
print(category_counts)

Number of FAQ entries per category: 
category
account    2
billing    3
general    1
dtype: int64


Question 6

In [11]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        matches = query_words.intersection(keywords)
        score = len(matches)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(matches),
                "confidence": score
            })

    result_df = pd.DataFrame(results)

    if result_df.empty:
        print("No matching FAQ found.")
        return result_df

    result_df = result_df.sort_values(
        by="confidence",
        ascending=False
    ).reset_index(drop=True)

    highest_score = result_df["confidence"].max()

    top_matches = result_df[
        result_df["confidence"] == highest_score
    ]

    print("Highest confidence score:", highest_score)

    if len(top_matches) > 1:
        print("Tie detected. All equally matching entries are:")
        display(top_matches)
    else:
        print("No tie. Best matching entry is:")
        display(top_matches)

    return result_df

In [12]:
tie_query = "fee"
tie_result = score_query_with_ties(tie_query, df)

Highest confidence score: 1
Tie detected. All equally matching entries are:


,question,answer,category,matched_keywords,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,fee,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,fee,1
2,how can i check my fee payment status,You can check your fee payment status from the...,billing,fee,1


In [13]:
non_tie_query = "password"
non_tie_result = score_query_with_ties(non_tie_query, df)

Highest confidence score: 1
No tie. Best matching entry is:


,question,answer,category,matched_keywords,confidence
0,how to reset password,Go to Settings > Reset Password.,account,password,1
